In [ ]:
import os, re, json, math, time
from collections import Counter
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import userdata, files

In [ ]:
API_KEY = userdata.get("OPENROUTER_API_KEY")
API_URL = "https://openrouter.ai/api/v1/chat/completions"

MODELS = {
    "gpt-4o-mini":           "openai/gpt-4o-mini",
    "gemini-3.1-flash-lite": "google/gemini-3.1-flash-lite",
    "grok-4.3":              "x-ai/grok-4.3",
    "qwen2.5-72b":           "qwen/qwen-2.5-72b-instruct",
    "llama-3.1-70b":         "meta-llama/llama-3.1-70b-instruct",
    "deepseek-v4-flash":     "deepseek/deepseek-v4-flash",
}

INPUT_PATH   = "med_calc_bangla_only.csv"
N_SAMPLE     = None   # None = all 677 rows. Staged runs: 6 (dry run) -> 100 -> None
SEED         = 42
N_RUNS       = 1
MAX_ATTEMPTS = 5      # no valid answer after this many attempts -> item excluded
MAX_WORKERS  = 8
MAX_TOKENS   = 8000   # reasoning tokens count toward this
TEMPERATURE  = 0
RAW_PATH     = "baseline_raw.jsonl"

In [ ]:
r = requests.get("https://openrouter.ai/api/v1/models",
                 headers={"Authorization": f"Bearer {API_KEY}"}, timeout=60)
available = {m["id"] for m in r.json()["data"]}
for name, slug in MODELS.items():
    if slug in available:
        print("OK     ", name, slug)
    else:
        key = slug.split("/")[1][:12]
        print("MISSING", name, slug, "-> similar:", sorted(a for a in available if key in a)[:6])

OK      gpt-4o-mini openai/gpt-4o-mini
OK      gemini-3.1-flash-lite google/gemini-3.1-flash-lite
OK      grok-4.3 x-ai/grok-4.3
OK      qwen2.5-72b qwen/qwen-2.5-72b-instruct
OK      llama-3.1-70b meta-llama/llama-3.1-70b-instruct
OK      deepseek-v4-flash deepseek/deepseek-v4-flash


In [ ]:
files.upload()   # choose med_calc_bangla_only.csv (skip if already uploaded this session)
df = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False)

STEP_PREFIX = re.compile(r"^\s*([০-৯0-9]+\s*[.।])")
def split_steps(t):
    return [ln.strip() for ln in str(t).split("\n") if ln.strip()]   # drops blank lines

df["_steps"] = df["Ground Truth Explanation"].apply(split_steps)
unnumbered = df["_steps"].apply(lambda s: not all(STEP_PREFIX.match(x) for x in s))
print(f"{int(unnumbered.sum())} rows have a line without a step number:", df.index[unnumbered].tolist())
assert (df["_steps"].str.len() >= 1).all()

sample = df.sample(frac=1, random_state=SEED)      # shuffled, so staged runs use nested subsets
if N_SAMPLE: sample = sample.head(N_SAMPLE)
print(len(sample), "rows |", sample["Calculator Name"].nunique(), "calculators")   # all: 677 | 38
sample.drop(columns="_steps").rename_axis("Source Index").to_csv("baseline_sample.csv", encoding="utf-8-sig")

Saving med_calc_bangla_only.csv to med_calc_bangla_only.csv
11 rows have a line without a step number: [657, 658, 661, 663, 664, 666, 667, 670, 672, 674, 676]
677 rows | 38 calculators


In [ ]:
PROMPT_TEMPLATE = """You are checking a medical-calculator solution for hallucinations. A hallucination is a step that states something unsupported by, or contradicting, the patient note, the question, or correct medical knowledge.
Hallucination types:
A (knowledge hallucination): the step states a wrong equation or scoring rule for this calculator.
B (extraction hallucination): the step states a wrong value or attribute for the patient, one that conflicts with the patient note.
C (computation hallucination): the step uses the correct formula and inputs but states a wrong arithmetic result.
If the solution contains no hallucination, set "hallucinated" to false and the other fields to null.

Patient note: {note}
Question: {question}
Explanation (numbered steps):
{explanation}

Return JSON only:
{{"hallucinated": true/false, "first_hallucinated_step": <int or null>, "type": "A"/"B"/"C"/null, "reason": "<one sentence>"}}"""

def call_model(slug, prompt, max_retries=3):
    """One attempt. Retries transient HTTP errors internally."""
    headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json",
               "X-Title": "MedCalc-BN hallucination baseline"}
    payload = {"model": slug, "messages": [{"role": "user", "content": prompt}],
               "max_tokens": MAX_TOKENS, "temperature": TEMPERATURE,
               "response_format": {"type": "json_object"}}
    last = ""
    for i in range(max_retries):
        try:
            r = requests.post(API_URL, headers=headers, json=payload, timeout=300)
        except requests.RequestException as e:
            last = str(e); time.sleep(2 ** i * 2); continue
        if r.status_code == 400 and ("response_format" in payload or "temperature" in payload):
            payload.pop("response_format", None); payload.pop("temperature", None)
            last = r.text[:200]; continue          # retry without optional params
        if r.status_code in (429, 500, 502, 503, 504):
            last = str(r.status_code); time.sleep(2 ** i * 3); continue
        if not r.ok:
            return {"error": f"{r.status_code}: {r.text[:300]}"}
        data = r.json()
        if "choices" not in data:
            last = str(data.get("error", ""))[:200]; time.sleep(2 ** i * 3); continue
        ch = data["choices"][0]
        return {"content": ch["message"].get("content") or "",
                "finish_reason": ch.get("finish_reason"),
                "served_model": data.get("model"), "usage": data.get("usage", {})}
    return {"error": f"failed after retries: {last}"}

def to_bool(x):
    if isinstance(x, bool): return x
    if isinstance(x, str):
        s = x.strip().lower()
        if s in ("true", "yes"):  return True
        if s in ("false", "no"):  return False
    return None

def parse_verdict(text):
    try:
        t = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
        obj = json.loads(re.search(r"\{.*\}", t, re.S).group(0))
        h = to_bool(obj.get("hallucinated"))
        if h is None: return None
        step = obj.get("first_hallucinated_step")
        step = int(step) if isinstance(step, (int, float)) or (isinstance(step, str) and step.strip().isdigit()) else None
        typ = obj.get("type")
        typ = typ.strip().upper() if isinstance(typ, str) and typ.strip().upper() in {"A", "B", "C"} else None
        return {"hallucinated": h, "first_hallucinated_step": step, "type": typ,
                "reason": str(obj.get("reason", ""))[:500]}
    except Exception:
        return None

In [ ]:
def one_attempt(name, idx, run, attempt):
    row = sample.loc[idx]
    prompt = PROMPT_TEMPLATE.format(note=row["Patient Note"], question=row["Question"],
                                    explanation="\n".join(row["_steps"]))
    res = call_model(MODELS[name], prompt)
    rec = {"model": name, "slug": MODELS[name], "index": int(idx), "run": run, "attempt": attempt}
    if "error" in res:
        rec.update(status="api_error", error=res["error"])
    else:
        v = parse_verdict(res["content"])
        rec.update(status="ok" if v else "parse_fail", raw=res["content"][:3000],
                   finish_reason=res["finish_reason"], served_model=res["served_model"],
                   usage=res["usage"], **(v or {}))
    return rec

def work(task):
    name, idx, run, prior = task
    recs = []
    for attempt in range(prior + 1, MAX_ATTEMPTS + 1):
        rec = one_attempt(name, idx, run, attempt)
        recs.append(rec)
        if rec["status"] == "ok":
            break
        time.sleep(min(2 * attempt, 10))
    return recs

# what is already done (survives Colab restarts as long as RAW_PATH still exists)
ok_done, attempts = set(), Counter()
if os.path.exists(RAW_PATH):
    for line in open(RAW_PATH, encoding="utf-8"):
        d = json.loads(line); k = (d["model"], d["index"], d["run"])
        attempts[k] += 1
        if d["status"] == "ok": ok_done.add(k)

tasks = []
for n in MODELS:
    for i in sample.index:
        for r in range(N_RUNS):
            k = (n, int(i), r)
            if k not in ok_done and attempts[k] < MAX_ATTEMPTS:
                tasks.append((n, int(i), r, attempts[k]))
print(f"{len(tasks)} items to run")

with open(RAW_PATH, "a", encoding="utf-8") as f, ThreadPoolExecutor(MAX_WORKERS) as ex:
    futs = [ex.submit(work, t) for t in tasks]
    for i, fut in enumerate(as_completed(futs), 1):
        for rec in fut.result():
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        f.flush()
        if i % 25 == 0 or i == len(futs): print(f"{i}/{len(futs)}")

4062 items to run
25/4062
50/4062
75/4062
100/4062
125/4062
150/4062
175/4062
200/4062
225/4062
250/4062
275/4062
300/4062
325/4062
350/4062
375/4062
400/4062
425/4062
450/4062
475/4062
500/4062
525/4062
550/4062
575/4062
600/4062
625/4062
650/4062
675/4062
700/4062
725/4062
750/4062
775/4062
800/4062
825/4062
850/4062
875/4062
900/4062
925/4062
950/4062
975/4062
1000/4062
1025/4062
1050/4062
1075/4062
1100/4062
1125/4062
1150/4062
1175/4062
1200/4062
1225/4062
1250/4062
1275/4062
1300/4062
1325/4062
1350/4062
1375/4062
1400/4062
1425/4062
1450/4062
1475/4062
1500/4062
1525/4062
1550/4062
1575/4062
1600/4062
1625/4062
1650/4062
1675/4062
1700/4062
1725/4062
1750/4062
1775/4062
1800/4062
1825/4062
1850/4062
1875/4062
1900/4062
1925/4062
1950/4062
1975/4062
2000/4062
2025/4062
2050/4062
2075/4062
2100/4062
2125/4062
2150/4062
2175/4062
2200/4062
2225/4062
2250/4062
2275/4062
2300/4062
2325/4062
2350/4062
2375/4062
2400/4062
2425/4062
2450/4062
2475/4062
2500/4062
2525/4062
2550/4062
2575

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0: return float("nan"), float("nan")
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

key = ["model", "index", "run"]
res = pd.DataFrame([json.loads(l) for l in open(RAW_PATH, encoding="utf-8")])
res = res[res["index"].isin(set(sample.index.astype(int)))]
valid = res[res.status == "ok"].drop_duplicates(key, keep="first")

# one row per (model, item, run), with its final state
items = pd.DataFrame([(n, int(i), r) for n in MODELS for i in sample.index for r in range(N_RUNS)], columns=key)
att = res.groupby(key).size().rename("attempts").reset_index()
items = items.merge(att, on=key, how="left").fillna({"attempts": 0})
items = items.merge(valid[key + ["hallucinated"]], on=key, how="left")
items["state"] = "pending"
items.loc[items["hallucinated"].notna(), "state"] = "valid"
items.loc[items["hallucinated"].isna() & (items["attempts"] >= MAX_ATTEMPTS), "state"] = "excluded"

# items that every model answered validly, for a like-for-like comparison
vk = {m: set(zip(g.loc[g.state == "valid", "index"], g.loc[g.state == "valid", "run"]))
      for m, g in items.groupby("model")}
common = set.intersection(*vk.values())

rows = []
for name, g in items.groupby("model", sort=False):
    v = g[g.state == "valid"]
    n_valid, correct = len(v), int((v["hallucinated"] == False).sum())
    lo, hi = wilson(correct, n_valid)
    vc = v[[(i, r) in common for i, r in zip(v["index"], v["run"])]]
    rows.append({
        "model": name,
        "items": len(g),
        "valid": n_valid,
        "excluded": int((g.state == "excluded").sum()),
        "pending": int((g.state == "pending").sum()),
        "correct": correct,
        "accuracy": f"{correct / n_valid:.1%}" if n_valid else None,
        "95% CI": f"{lo:.1%}-{hi:.1%}",
        "accuracy (common items)": f"{(vc['hallucinated'] == False).mean():.1%}" if len(vc) else None,
        "avg attempts": round(g.loc[g.attempts > 0, "attempts"].mean(), 2),
        "truncated": int((res[res.model == name].finish_reason == "length").sum()),
    })
summary = pd.DataFrame(rows).sort_values("correct", ascending=False)
print(summary.to_string(index=False))
print(f"\nItems answered validly by all models: {len(common)} of {len(sample) * N_RUNS}")
if (summary["pending"] > 0).any():
    print("Some items are still pending (run interrupted). Re-run Cell 6.")

flagged = valid[valid.hallucinated == True]
print("\nType named by the model (among flags):")
print(pd.crosstab(flagged["model"], flagged["type"].fillna("none")))

usage = res["usage"].apply(lambda u: u if isinstance(u, dict) else {})
res["prompt_tokens"] = usage.apply(lambda u: u.get("prompt_tokens", 0))
res["completion_tokens"] = usage.apply(lambda u: u.get("completion_tokens", 0))
print("\nTokens per model (all attempts):")
print(res.groupby("model")[["prompt_tokens", "completion_tokens"]].sum())
print("\nServed model IDs:", res.groupby("model")["served_model"].agg(lambda s: sorted(set(s.dropna()))).to_dict())

# items excluded after MAX_ATTEMPTS failed attempts
cols = [c for c in ["status", "error", "finish_reason", "raw"] if c in res.columns]
last = res.sort_values("attempt").groupby(key).tail(1)[key + cols]
excluded = items[items.state == "excluded"][key].merge(last, on=key, how="left")
excluded.to_csv("excluded_items.csv", index=False, encoding="utf-8-sig")
print(f"\nExcluded items: {len(excluded)} (saved to excluded_items.csv)")

                model  items  valid  excluded  pending  correct accuracy      95% CI accuracy (common items)  avg attempts  truncated
          qwen2.5-72b    677    677         0        0      543    80.2% 77.0%-83.0%                   80.4%           1.0          0
             grok-4.3    677    677         0        0      513    75.8% 72.4%-78.9%                   76.2%           1.0          0
        llama-3.1-70b    677    677         0        0      512    75.6% 72.3%-78.7%                   75.9%           1.0          0
    deepseek-v4-flash    677    672         5        0      454    67.6% 63.9%-71.0%                   67.6%           1.1         67
gemini-3.1-flash-lite    677    677         0        0      292    43.1% 39.4%-46.9%                   43.5%           1.0          1
          gpt-4o-mini    677    677         0        0       94    13.9% 11.5%-16.7%                   14.0%           1.0          0

Items answered validly by all models: 672 of 677

Type named 

In [ ]:
out = []
for idx, g in flagged.groupby("index"):
    src = sample.loc[idx]; steps = src["_steps"]
    modes = g["first_hallucinated_step"].dropna().mode()
    step = int(modes.iloc[0]) if len(modes) else None
    out.append({
        "Source Index": idx,
        "Calculator Name": src["Calculator Name"],
        "n_models_flagged": g["model"].nunique(),
        "models": ", ".join(sorted(g["model"].unique())),
        "details": "\n".join(f"{r.model}: step {r.first_hallucinated_step}, type {r.type} - {r.reason}" for r in g.itertuples()),
        "most_flagged_step": step,
        "most_flagged_step_text": steps[step - 1] if step and 1 <= step <= len(steps) else "",
        "Patient Note": src["Patient Note"],
        "Question": src["Question"],
        "Explanation": "\n".join(steps),
    })
review = pd.DataFrame(out).sort_values("n_models_flagged", ascending=False)
print(len(review), "items flagged by at least one model;", int((review.n_models_flagged >= 2).sum()), "by two or more")
review.to_csv("baseline_flagged_for_review.csv", index=False, encoding="utf-8-sig")

620 items flagged by at least one model; 466 by two or more


In [ ]:
summary.to_csv("baseline_summary.csv", index=False)
for f in ["baseline_summary.csv", "baseline_flagged_for_review.csv",
          "excluded_items.csv", "baseline_raw.jsonl", "baseline_sample.csv"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>